# Credit Risk Modeling — Stage 1: Data Preparation
### Basel II–Aligned PD Scorecard Pipeline | LendingClub Consumer Loan Data

---
**Project:** End-to-End Credit Risk Model (PD / LGD / EAD / Expected Loss)  
**Stage:** 1 of 4 — Data Preparation  
**Dataset:** LendingClub Loan Data (2007–2018)  
**Author:** Harsh  
**Framework:** Basel II Internal Ratings-Based (IRB) Approach  

---

## Executive Summary

This notebook establishes a model-ready dataset for Probability of Default (PD) estimation using LendingClub consumer loan data. The objective is to transform raw transactional and behavioral data into a clean, non-leaking, temporally consistent dataset suitable for downstream WOE-based feature engineering and logistic regression modeling.

Key deliverables include:

- Binary target definition aligned with regulatory credit risk practices
- Strict removal of post-origination leakage variables
- Time-based data partitioning (Train / Test / Out-of-Time)
- Economically consistent missing value treatment strategy
- Controlled outlier handling (winsorization)

This stage ensures that the model learns only from information available at the time of underwriting, preserving real-world deployability.

## Business Context

In retail credit risk modeling, the central objective is to estimate:

$$PD = P(\text{Default within horizon} \mid \text{Borrower characteristics at origination})$$

A critical requirement is that no future information contaminates the model, as this would inflate model performance but fail in production.

For example:

- `total_pymnt` reflects realized cashflows → not available at underwriting
- `recoveries` reflects post-default behavior → direct leakage of target

Thus, Stage 1 is fundamentally about **causal integrity** of the dataset.

## Methodology

The data preparation follows a structured risk modeling framework:

1. Target Definition
2. Filtering Relevant Observations
3. Time Variable Construction
4. Leakage Removal
5. Time-Based Dataset Split
6. Feature Filtering & Structuring
7. Missing Value Treatment (Economic Logic Driven)
8. Outlier Handling (Winsorization)

Each step is implemented with explicit economic rationale.

## Step 1: Load Data and Initial Inspection

In [ ]:
import numpy as np
import pandas as pd

pd.options.display.max_columns = None

df_raw = pd.read_csv(r'loan.csv', low_memory=False)
df1 = df_raw.copy()

print("Shape:", df1.shape)
print(df1['loan_status'].value_counts(normalize=True))

**Interpretation**

Initial inspection shows class imbalance typical of credit portfolios (low default rate). This is expected and will later influence model calibration and evaluation.

## Step 2: Target Definition (Good vs Bad)

In [ ]:
good_status = [
    'Fully Paid',
    'Does not meet the credit policy. Status:Fully Paid'
]

bad_status = [
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Charged Off'
]

df1 = df1[df1['loan_status'].isin(good_status + bad_status)].copy()

df1['target'] = np.where(df1['loan_status'].isin(bad_status), 0, 1)

print(df1['target'].value_counts(normalize=True))

**Interpretation**

- Target = 1 → Good (Non-default)
- Target = 0 → Bad (Default)

This aligns with scorecard convention where higher score = lower risk.

Economically:
- "Charged Off" implies loss realization → true default proxy
- "Fully Paid" implies survival → non-default

## Step 3: Time Variable Construction

In [ ]:
df1['issue_d_dt'] = pd.to_datetime(df1['issue_d'], format='%b-%Y', errors='coerce')

print(df1.groupby(df1['issue_d_dt'].dt.year)['target'].agg(['count','mean']))

**Interpretation**

Time dimension is critical because:

- Credit cycles (boom vs downturn) affect default rates
- Model must generalize across time → not just cross-sectional

## Step 4: Leakage Removal (Critical Step)

In [ ]:
leak_cols = [
    'out_prncp','out_prncp_inv',
    'total_pymnt','total_pymnt_inv',
    'total_rec_prncp','total_rec_int','total_rec_late_fee',
    'recoveries','collection_recovery_fee',
    'last_pymnt_d','last_pymnt_amnt',
    'next_pymnt_d'
]

future_cols = [
    'last_credit_pull_d',
    'hardship_flag','hardship_type','hardship_reason','hardship_status',
    'debt_settlement_flag','debt_settlement_flag_date',
    'settlement_status','settlement_date'
]

drop_cols = [
    'id','member_id','url','desc','title','zip_code',
    'emp_title','pymnt_plan'
]

df1.drop(columns=leak_cols + future_cols + drop_cols, inplace=True, errors='ignore')

**Interpretation**

This is the most important step in credit modeling.

Example: `recoveries > 0` ⇒ loan has already defaulted → model would "cheat"

This ensures model predictions simulate decision-time underwriting, not hindsight.

## Step 5: Time-Based Data Split (Train / Test / OOT)

In [ ]:
df1 = df1.sort_values('issue_d_dt')

train = df1[df1['issue_d_dt'] < '2014-01-01']
test  = df1[(df1['issue_d_dt'] >= '2014-01-01') & (df1['issue_d_dt'] < '2015-01-01')]
oot   = df1[df1['issue_d_dt'] >= '2015-01-01']

X_train = train.drop(columns=['target'])
y_train = train['target']

X_test  = test.drop(columns=['target'])
y_test  = test['target']

X_oot   = oot.drop(columns=['target'])
y_oot   = oot['target']

**Interpretation**

- Train → model estimation
- Test  → validation
- OOT (Out-of-Time) → stability check

OOT is crucial because a model that works only in-sample is not usable in banking.

## Step 6: Feature Structuring

### Term & Employment Length

In [ ]:
for d in (X_train, X_test, X_oot):
    d['term_int'] = d['term'].str.replace(' months','').astype(float)

def clean_emp(df):
    df['emp_length_clean'] = df['emp_length']
    df['emp_length_clean'] = df['emp_length_clean'].str.replace('+ years','',regex=False)
    df['emp_length_clean'] = df['emp_length_clean'].str.replace('< 1 year','0',regex=False)
    df['emp_length_clean'] = df['emp_length_clean'].str.replace(' years','',regex=False)
    df['emp_length_int'] = pd.to_numeric(df['emp_length_clean'], errors='coerce')

clean_emp(X_train)
clean_emp(X_test)
clean_emp(X_oot)

**Interpretation**

- `term` → numeric for modeling
- `emp_length` → proxy for income stability

Longer employment typically reduces default risk due to income consistency.

## Step 7: Credit History Transformation

In [ ]:
for d in (X_train, X_test, X_oot):
    d['earliest_cr_line_dt'] = pd.to_datetime(
        d['earliest_cr_line'],
        format='%b-%Y',
        errors='coerce'
    )

ref_date = X_train['issue_d_dt'].max()

for d in (X_train, X_test, X_oot):
    d['mths_since_earliest_cr_line'] = ((ref_date - d['earliest_cr_line_dt']) / np.timedelta64(1,'D') / 30).round()
    d.loc[d['mths_since_earliest_cr_line'] < 0, 'mths_since_earliest_cr_line'] = 0

**Interpretation**

This converts raw dates into behavioral credit variables:

- Longer credit history → lower uncertainty → lower PD
- Negative values corrected → data quality control

## Step 8: Missing Value Treatment (Economic Logic Driven)

### Categorical Variables

In [ ]:
# Categorical
for col in X_train.select_dtypes(include='object').columns:
    X_train[col] = X_train[col].fillna('MISSING')
    X_test[col]  = X_test[col].fillna('MISSING')
    X_oot[col]   = X_oot[col].fillna('MISSING')

### Numerical Treatment Strategy

In [ ]:
# Time-based missing → 999
time_vars = ['mths_since_last_delinq']

for col in time_vars:
    for d in (X_train, X_test, X_oot):
        if col in d.columns:
            d[col] = d[col].fillna(999)

# Structural missing → -1
structural_vars = ['total_rev_hi_lim']

for col in structural_vars:
    for d in (X_train, X_test, X_oot):
        if col in d.columns:
            d[col] = d[col].fillna(-1)

# Remaining → median
for col in X_train.select_dtypes(include=['int64','float64']).columns:
    median = X_train[col].median()
    for d in (X_train, X_test, X_oot):
        d[col] = d[col].fillna(median)

**Interpretation**

This is not generic imputation:

- `999` → "no delinquency observed" (behavioral signal)
- `-1`  → "no credit history / thin file"
- Median used only where missing has no economic meaning

This preserves predictive signal for WOE transformation later.

## Step 9: Winsorization (Outlier Treatment)

In [ ]:
winsor_cols = [
    'annual_inc', 'loan_amnt', 'installment',
    'dti', 'int_rate'
]

for col in winsor_cols:
    lower = X_train[col].quantile(0.01)
    upper = X_train[col].quantile(0.99)

    for d in (X_train, X_test, X_oot):
        d[col] = d[col].clip(lower, upper)

**Interpretation**

Extreme values distort:

- Logistic regression coefficients
- WOE bin stability

Example: Extremely high income may not reduce risk proportionally → capped to maintain monotonic relationship.

## Results / Outputs

Clean dataset with:

- No leakage variables
- Time-consistent splits
- Economically meaningful missing treatment

Final datasets: `X_train`, `X_test`, `X_oot`, `y_train`, `y_test`, `y_oot`

At this stage:

- The dataset reflects true underwriting conditions
- Variables are aligned with credit risk theory
- Data is ready for WOE transformation and binning

Critically: **the model will now learn borrower risk, not loan outcomes.**

## Limitations

Despite robust preparation, several limitations remain:

**Point-in-Time (PIT) bias:** No macroeconomic variables (GDP, unemployment) — the model is calibrated to historical conditions only.

**No reject inference:** Only accepted applicants are modeled. Borrowers who were declined are unobserved, introducing selection bias.

**LGD/EAD not yet incorporated:** Current dataset focuses only on PD.

**Behavioral variables limited:** No dynamic repayment history used.

**Static snapshot:** Does not capture borrower evolution over time.

## Transition to Stage 2 — Feature Engineering

This concludes Stage 1: Data Preparation. The six output DataFrames (`X_train`, `X_test`, `X_oot`, `y_train`, `y_test`, `y_oot`) are clean, causally sound, and temporally consistent.

**Stage 2** will perform WOE (Weight of Evidence) and IV (Information Value) based feature engineering — fine classing, coarse classing, monotonicity validation, and multicollinearity assessment via VIF — to produce the final WOE-transformed feature set for logistic regression.

---
*Next notebook: `02_feature_engineering.ipynb`*